# ⚽ FIFA World Cup 2026 — Modèle Niveau 1
## Prédiction par match : Victoire / Nul / Défaite
**FrenchTeam — Wild Code School — Juin 2026**

## 📦 Étape 1 — Imports

In [ ]:
# Importer les bibliothèques nécessaires
import pandas as pd
# numpy pour les calculs numériques
import numpy as np

print('✅ Imports OK')

## 📂 Étape 2 — Chargement des données

In [ ]:
# Charger le dataset des résultats historiques
df = pd.read_csv('../data/results.csv')
# Afficher la taille : (nombre de lignes, nombre de colonnes)
print(df.shape)
df.head(10)

## 🔍 Étape 3 — Exploration (EDA)
*(à compléter)*

In [ ]:
# Vérifier les types de données et les valeurs manquantes
print(df.dtypes)
# Séparation visuelle
print("─" * 40)
# Compter les valeurs manquantes par colonne
print(df.isnull().sum())

In [ ]:
# Voir tous les types de tournois disponibles
print(df['tournament'].value_counts())

## 🧹 Étape 4 — Nettoyage & Feature Engineering
*(à compléter)*

In [ ]:
# Filtrer uniquement CdM + Qualifications
df_filtre = df[df['tournament'].isin(['FIFA World Cup', 'FIFA World Cup qualification'])]

# Supprimer les lignes avec des scores manquants
df_filtre = df_filtre.dropna(subset=['home_score', 'away_score'])

# Vérifier le résultat
print(f"Nombre de matchs retenus : {len(df_filtre)}")

In [ ]:
# Convertir la date en datetime
df_filtre['date'] = pd.to_datetime(df_filtre['date'])

# Supprimer les matchs avant 1990
df_filtre = df_filtre[df_filtre['date'] >= '1990-01-01']

# Vérifier
print(f"Matchs après 1990 : {len(df_filtre)}")

In [ ]:
# Créer la variable cible : résultat du match
# Si home_score > away_score → Victoire domicile (V)
# Si home_score < away_score → Défaite domicile (D)
# Si home_score == away_score → Nul (N)
def get_resultat(row):
    if row['home_score'] > row['away_score']:
        return 'V'
    elif row['home_score'] < row['away_score']:
        return 'D'
    else:
        return 'N'

# Appliquer la fonction sur chaque ligne
df_filtre['resultat'] = df_filtre.apply(get_resultat, axis=1)

# Vérifier la distribution des résultats
print(df_filtre['resultat'].value_counts())

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Créer un encodeur pour les noms d'équipes
encodeur = LabelEncoder()

# Encoder home_team et away_team en nombres
df_filtre['home_team_enc'] = encodeur.fit_transform(df_filtre['home_team'])
df_filtre['away_team_enc'] = encodeur.fit_transform(df_filtre['away_team'])

# Vérifier le résultat
print(df_filtre[['home_team', 'home_team_enc', 'away_team', 'away_team_enc']].head())

In [ ]:
# Charger le classement FIFA le plus récent
df_ranking = pd.read_csv('../data/fifa_ranking-2024-06-20.csv')

# Voir les premières lignes
print(df_ranking.shape)
df_ranking.head()

In [ ]:
# 7. Corriger les noms d'équipes pour correspondre au CSV FIFA
corrections = {'United States': 'USA', 'South Korea': 'Korea Republic', 'Iran': 'IR Iran', 'Ivory Coast': "Côte d'Ivoire", 'DR Congo': 'Congo DR', 'Czech Republic': 'Czechia'}

# Supprimer Cape Verde de la liste (absent du CSV FIFA)
equipes_2026 = [e for e in equipes_2026 if e != 'Cape Verde']

# Appliquer les corrections
equipes_2026_corrigees = [corrections.get(e, e) for e in equipes_2026]

# 8. Filtrer les 47 équipes + garder le rang le plus récent
df_ranking_2026 = df_ranking[df_ranking['country_full'].isin(equipes_2026_corrigees)].sort_values('rank_date', ascending=False).drop_duplicates(subset='country_full', keep='first')

# 9. Vérifier
print(f"Équipes trouvées : {len(df_ranking_2026)}")
print(f"Rangs manquants : {df_ranking_2026['rank'].isna().sum()}")
df_ranking_2026[['country_full', 'rank', 'total_points']].sort_values('total_points', ascending=False).head(10)

In [ ]:
# Garder uniquement les colonnes utiles du classement FIFA
df_ranking_clean = df_ranking_2026[['country_full', 'total_points']].copy()

# Renommer pour la fusion
df_ranking_clean.columns = ['team', 'fifa_points']

# Vérifier
print(f"NaN dans fifa_points : {df_ranking_clean['fifa_points'].isna().sum()}")
df_ranking_clean.sort_values('fifa_points', ascending=False).head(10)

In [ ]:
# Fusionner le classement FIFA avec les matchs
# Pour l'équipe à domicile
df_filtre = df_filtre.merge(df_ranking_clean.rename(columns={'team': 'home_team', 'fifa_points': 'home_fifa_points'}), on='home_team', how='left')

# Pour l'équipe à l'extérieur
df_filtre = df_filtre.merge(df_ranking_clean.rename(columns={'team': 'away_team', 'fifa_points': 'away_fifa_points'}), on='away_team', how='left')

# Vérifier
print(f"Colonnes disponibles : {df_filtre.columns.tolist()}")
print(f"NaN home_fifa_points : {df_filtre['home_fifa_points'].isna().sum()}")
print(f"NaN away_fifa_points : {df_filtre['away_fifa_points'].isna().sum()}")

In [ ]:
# Remplacer les NaN par la moyenne des points FIFA
moyenne_points = df_ranking_clean['fifa_points'].mean()
df_filtre['home_fifa_points'] = df_filtre['home_fifa_points'].fillna(moyenne_points)
df_filtre['away_fifa_points'] = df_filtre['away_fifa_points'].fillna(moyenne_points)

# Vérifier
print(f"Moyenne FIFA points : {moyenne_points:.2f}")
print(f"NaN restants home : {df_filtre['home_fifa_points'].isna().sum()}")
print(f"NaN restants away : {df_filtre['away_fifa_points'].isna().sum()}")

In [ ]:
# Calculer la forme récente de chaque équipe (5 derniers matchs)
def forme_recente(team, date, df):
    # Matchs joués par cette équipe avant cette date
    matchs = df[((df['home_team'] == team) | (df['away_team'] == team)) & (df['date'] < date)].tail(5)
    if len(matchs) == 0:
        return 0.5  # valeur neutre si pas d'historique
    points = 0
    for _, row in matchs.iterrows():
        if row['home_team'] == team and row['resultat'] == 'V':
            points += 3
        elif row['away_team'] == team and row['resultat'] == 'D':
            points += 3
        elif row['resultat'] == 'N':
            points += 1
    return points / (len(matchs) * 3)  # normaliser entre 0 et 1

# Appliquer sur toutes les lignes (peut prendre 1-2 min)
print("Calcul forme récente en cours...")
df_filtre['home_forme'] = df_filtre.apply(lambda row: forme_recente(row['home_team'], row['date'], df_filtre), axis=1)
df_filtre['away_forme'] = df_filtre.apply(lambda row: forme_recente(row['away_team'], row['date'], df_filtre), axis=1)
print("✅ Forme récente calculée !")
print(df_filtre[['home_team', 'away_team', 'home_forme', 'away_forme']].head(5))

In [ ]:
# Moyenne de buts marqués et encaissés par équipe
buts_home = df_filtre.groupby('home_team')['home_score'].mean().rename('moy_buts_marques_home')
buts_away = df_filtre.groupby('away_team')['away_score'].mean().rename('moy_buts_marques_away')
encaisses_home = df_filtre.groupby('home_team')['away_score'].mean().rename('moy_buts_encaisses_home')
encaisses_away = df_filtre.groupby('away_team')['home_score'].mean().rename('moy_buts_encaisses_away')

# Fusionner avec df_filtre
df_filtre = df_filtre.merge(buts_home, on='home_team', how='left')
df_filtre = df_filtre.merge(buts_away, on='away_team', how='left')
df_filtre = df_filtre.merge(encaisses_home, on='home_team', how='left')
df_filtre = df_filtre.merge(encaisses_away, on='away_team', how='left')

# Vérifier
print(df_filtre[['home_team', 'away_team', 'moy_buts_marques_home', 'moy_buts_marques_away']].head(5))

In [ ]:
# Face-à-face historique entre les deux équipes
def face_a_face(home, away, date, df):
    # Matchs entre ces deux équipes avant cette date
    matchs = df[((df['home_team'] == home) & (df['away_team'] == away)) | ((df['home_team'] == away) & (df['away_team'] == home))]
    matchs = matchs[matchs['date'] < date].tail(10)
    if len(matchs) == 0:
        return 0.5  # valeur neutre
    victoires_home = 0
    for _, row in matchs.iterrows():
        if (row['home_team'] == home and row['resultat'] == 'V') or (row['away_team'] == home and row['resultat'] == 'D'):
            victoires_home += 1
    return victoires_home / len(matchs)

# Appliquer (peut prendre 2-3 min)
print("Calcul face-à-face en cours...")
df_filtre['h2h'] = df_filtre.apply(lambda row: face_a_face(row['home_team'], row['away_team'], row['date'], df_filtre), axis=1)
print(f"✅ Face-à-face calculé !")
print(df_filtre[['home_team', 'away_team', 'h2h']].head(5))

## 🤖 Étape 5 — Modèle ML
*(à compléter)*

In [ ]:
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder

# Encoder la cible en nombres (XGBoost ne comprend pas V/N/D)
enc_y = LabelEncoder()
y_enc = enc_y.fit_transform(y)

# Séparer train et test
X_train, X_test, y_train, y_test = train_test_split(X, y_enc, test_size=0.2, random_state=42)

# Modèle XGBoost
modele = XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=4, random_state=42, eval_metric='mlogloss')
modele.fit(X_train, y_train)

# Prédire et évaluer
y_pred = modele.predict(X_test)
print(f"Accuracy XGBoost : {accuracy_score(y_test, y_pred):.2%}")
print(classification_report(y_test, y_pred, target_names=enc_y.classes_))

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# Nouvelles features avec classement FIFA
# Ajouter diff_fifa_points comme nouvelle feature
# Features avec forme récente
X = df_filtre[['home_team_enc', 'away_team_enc', 'home_fifa_points', 'away_fifa_points', 'home_forme', 'away_forme', 'moy_buts_marques_home', 'moy_buts_marques_away', 'moy_buts_encaisses_home', 'moy_buts_encaisses_away', 'h2h']]
y = df_filtre['resultat']

# Séparer train (80%) et test (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Entraîner le modèle
modele = RandomForestClassifier(n_estimators=100, random_state=42)
modele.fit(X_train, y_train)

# Prédire et évaluer
y_pred = modele.predict(X_test)
print(f"Accuracy : {accuracy_score(y_test, y_pred):.2%}")
print(classification_report(y_test, y_pred))

In [ ]:
# Créer un tableau détaillé des prédictions
import pandas as pd

# Récupérer les équipes du jeu de test
df_test = df_filtre.iloc[X_test.index][['home_team', 'away_team', 'home_score', 'away_score', 'resultat']].copy()

# Ajouter les prédictions
df_test['prediction'] = y_pred

# Ajouter si correct ou non
df_test['correct'] = df_test['resultat'] == df_test['prediction']

# Afficher les résultats
print(f"✅ Prédictions correctes : {df_test['correct'].sum()}")
print(f"❌ Prédictions incorrectes : {(~df_test['correct']).sum()}")
print(f"📊 Accuracy : {df_test['correct'].mean():.2%}")

# Afficher un échantillon
df_test.head(20)

## 📊 Étape 6 — Sauvegarde
*(à compléter)*

In [ ]:
# ── SAUVEGARDE DU MODÈLE ─────────────────────────────────────────
import joblib

# Sauvegarder le modèle XGBoost
joblib.dump(modele, '../models/modele_niveau1_final.pkl')

# Sauvegarder l'encodeur des équipes
joblib.dump(encodeur, '../models/encodeur_equipes.pkl')

# Sauvegarder l'encodeur de la cible (V/N/D → 0/1/2)
joblib.dump(enc_y, '../models/encodeur_cible.pkl')

# Sauvegarder le classement FIFA propre
df_ranking_clean.to_csv('../data/ranking_2026.csv', index=False)

print("✅ Modèle sauvegardé !")
print("✅ Encodeurs sauvegardés !")
print("✅ Ranking FIFA sauvegardé !")